# Exam Project - Introduction to Social Data Science
August 28, 2024

## Project: Forecasting Vote Counts for Danish Borgerforslag

## Group 7:
- Oliver Nyrop Weeks (vsn684)
- Sofus Galavits Møller (qvc730)
- Victor V. Kristensen (gcp458)
- Jonas T. Schmidt (mcp656)

## Load modules

In [10]:
# Module Imports
import selenium                # For navigating borgerforslag.dk
import requests                # For web scraping
import pandas as pd            # For data manipulation and analysis
from bs4 import BeautifulSoup  # For parsing HTML content
import time                    # For managing time delays during scraping
import tqdm                    # For displaying progress bars during scraping
import random                  # For randomizing delays in scraping to avoid detection
import pprint                  # For neatly displaying JSON code
import re                      # For pattern recognition in extracted HTML
from pathlib import Path       # For handling file paths
import csv                     # For exporting data to .csv files
import json                    # For exporting data to .json files
import os                      # For interacting with the operating system (e.g., file paths, environment variables)

# Class and Function Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By                # For using CSS selectors (e.g., cookie-clicking)
from selenium.webdriver.support.ui import WebDriverWait    # For implementing explicit waits
from selenium.webdriver.support import expected_conditions as EC 
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.keys import Keys            # For simulating keyboard actions (e.g., RETURN key)
from selenium.common.exceptions import NoSuchElementException


## Connecting to Borgerforslag.dk

We first need to connect to Borgerforslag.dk to gather our data. This is accomplished using Selenium. We use the Selenium Chrome Driver to navigate the website, automating tasks such as selecting the correct site path and accepting cookies. Additionally, we adjust the settings to display all proposals, including the expired ones.

In [11]:
# Set Chrome options to disable the search engine choice screen
chrome_options = Options()
chrome_options.add_argument("--disable-search-engine-choice-screen")

# Initialize the Selenium Chrome driver with the specified options
driver = webdriver.Chrome(options=chrome_options)

# URL for Borgerforslag.dk
url_Borgerforslag = "https://borgerforslag.dk/"

# Use the .get() method to open the URL in the Selenium Chrome browser
driver.get(url_Borgerforslag)

# Attempt to locate and click the cookie consent button
try:
    cookie = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'CybotCookiebotDialogBodyLevelButtonLevelOptinAllowallSelection'))
    )
    cookie.click()
except TimeoutException:
    print("Element not found within the specified wait time.")

# Attempt to select "Alle" instead of "Igangværende" proposals
try:
    # Locate and click the filter dropdown
    click_1 = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--value-item'))
    )
    click_1.click()

    # Locate and select the "Alle" option
    click_2 = WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--option-0'))
    )
    click_2.click()
except TimeoutException:
    print("Element not found within the specified wait time.")


We have now connected to the site, accepted cookies, and selected all proposals instead of just the active ones. We will now scroll (and click) through the site to be able to view all the elements at once. We create a loop that clicks through this, and stops when no more borgerforslag can be loaded (when the "load more" button disappears).

In [12]:
# Initialize a flag to control the loop
alarm = False  # Defining a stop_alarm

# Loop to navigate through additional pages
while not alarm:
    try:
        # Scroll to the bottom of the page to make sure the "load more" button is visible
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        # Wait for the "load more" button to be clickable
        wait = WebDriverWait(driver, 10)
        load_more_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button.dFsu8t.fYY1lZ.vFact_DoNotReadAloud._3-IJkM._3CrCss')))
        
        # Click the "load more" button
        load_more_button.click()

        # Introduce a random delay between clicks to mimic human interaction
        time.sleep(random.uniform(1, 2))
    
    except (NoSuchElementException, TimeoutException):
        # If the button is not found or not clickable, stop the loop
        alarm = True
        print("No more pages to go through or button not found")

print("Finished scrolling through all pages")


No more pages to go through or button not found
Finished scrolling through all pages


We have now navigated to the bottom of the page in our Selenium Chrome browser, successfully loading all of the borgerforslag. This means we can begin the scraping process, which will collect all of the HTML, including the individual links to each borgerforslag.

In [31]:
# Define the directory and file name
output_dir = 'Scraping_files'  # Corrected directory name
os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist
output_file_path = os.path.join(output_dir, 'test.txt')

# Parse the page source with BeautifulSoup using 'lxml' parser
soup = BeautifulSoup(driver.page_source, 'lxml')

# Find all <a> tags with the specified class
all_sites = soup.find_all('a', class_='lQq327')

# Extract the href attributes from each <a> tag and store them in a list
links = [site['href'] for site in all_sites][0:50]

# Print the number of borgerforslag found
print(f"Number of borgerforslag found: {len(links)}\n")

# Save the list of links to the file for later checks
with open(output_file_path, 'w') as file:
    for link in links:
        file.write(link + '\n')

# Confirm that the file has been saved
print(f"List of links saved to {output_file_path}")


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=127.0.6533.120)
Stacktrace:
	GetHandleVerifier [0x00007FF75E219642+30946]
	(No symbol) [0x00007FF75E1CE3D9]
	(No symbol) [0x00007FF75E0C6FDA]
	(No symbol) [0x00007FF75E09CB85]
	(No symbol) [0x00007FF75E1437A7]
	(No symbol) [0x00007FF75E15A771]
	(No symbol) [0x00007FF75E13C813]
	(No symbol) [0x00007FF75E10A6E5]
	(No symbol) [0x00007FF75E10B021]
	GetHandleVerifier [0x00007FF75E34F84D+1301229]
	GetHandleVerifier [0x00007FF75E35BDC7+1351783]
	GetHandleVerifier [0x00007FF75E352A13+1313971]
	GetHandleVerifier [0x00007FF75E24DD16+245686]
	(No symbol) [0x00007FF75E1D759F]
	(No symbol) [0x00007FF75E1D3814]
	(No symbol) [0x00007FF75E1D39A2]
	(No symbol) [0x00007FF75E1CA3FF]
	BaseThreadInitThunk [0x00007FF8948C7374+20]
	RtlUserThreadStart [0x00007FF8959DCC91+33]


Now that we have all the links to the borgerforslag, we can start scraping the necessary information from each proposal.

In the following block, we define a logging function to record key details about our scraping process for documentation purposes.

In [28]:
# Define the log function to gather and record log information
def log(response, logfile, url, run_id, output_path=os.getcwd()):
    # Open or create the log file
    if os.path.isfile(logfile):  # If the log file exists, open it for appending
        log = open(logfile, 'a')
    else:  # If the log file does not exist, create it with headers
        log = open(logfile, 'w')
        header = ['run_id', 'timestamp', 'status_code', 'length', 'url', 'output_file']
        log.write(';'.join(header) + "\n")  # Write headers and move to the next line
        
    # Gather log information
    status_code = response.status_code  # Status code from the HTTP response
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time()))  # Current local time
    length = len(response.text)  # Length of the HTML content
    
    # Append the gathered information to the log file
    with open(logfile, 'a') as log:
        log.write(f'{run_id};{timestamp};{status_code};{length};{url};{output_path}' + "\n")  # Log the details and move to a new line


With the logging function defined, we can now proceed to scrape data from the individual borgerforslag links. See the "Borgerforslag.ipynb" for next steps.

In [29]:
# Log file name
log_filename = os.path.join(output_dir, 'logfile_borgerforslag.csv')
raw_output_filename = os.path.join(output_dir, 'html_content_raw.txt')
normal_output_filename = os.path.join(output_dir, 'html_content_normal.txt')

# Generate a unique run ID, for example, based on the current time
run_id = time.strftime('%Y%m%d%H%M%S')

# List to store the HTML content from each URL
list_htmls = []

# Limit the range to the first 5 links for testing
test_links = links[:5]

# Loop through each URL in the links list
for i in tqdm.tqdm(links):
    try:
        # Send an HTTP GET request to the current URL with custom headers
        response = requests.get(i, headers={"Name": "Oliver Nyrop Weeks", "Email": "vsn684@alumni.ku.dk"})
        
        # Get the HTML content from the response
        html = response.text
        
        # Append the HTML content to the list
        list_htmls.append(html)
        
        # Log the request details, now including run_id
        log(response, log_filename, i, run_id)
        
        # Sleep for 1 to 2 seconds to avoid overloading the server
        time.sleep(random.uniform(1, 2))
    except Exception as e:
        # If an error occurs, print the URL and the error
        print(f"Error with URL: {i}")
        print(e)
        # Log the error with a status code of 0 (indicating failure)
        log(None, log_filename, i, run_id)

100%|██████████| 5/5 [00:08<00:00,  1.70s/it]


In [30]:
# Load the log data
log_df = pd.read_csv(log_filename, sep=';', header=0, names=['Run ID', 'Timestamp', 'Status Code', 'Size', 'URL', 'File Path'])

# Filter out any rows where the Timestamp is not in the expected format
log_df = log_df[log_df['Timestamp'].str.match(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}')]

# Convert the Timestamp to datetime
log_df['Timestamp'] = pd.to_datetime(log_df['Timestamp'], format='%Y-%m-%d %H:%M:%S')

# Extract the ID from the URL
log_df['ID'] = log_df['URL'].str.extract(r'Id=([A-Za-z0-9\-]+)')

# Define the path to the test.txt file in the Scraping_files directory
test_file_path = os.path.join('Scraping_files', 'test.txt')

# Step 1: Read the IDs from test.txt
with open(test_file_path, 'r') as file:
    test_urls = file.readlines()

# Extract IDs from the URLs in test.txt
test_ids = [url.split('Id=')[-1].strip() for url in test_urls]

# Step 2: Get the unique IDs from the latest log data (log_df)
log_ids = log_df['ID'].unique()

# Step 3: Find IDs that are in test.txt but not in the latest log data
missing_ids = set(test_ids) - set(log_ids)

# Step 4: Display the missing IDs
print("Missing IDs:")
for missing_id in missing_ids:
    print(missing_id)


Missing IDs:


We have now scraped all the data and stored in a list. 

We also save the data physically:

In [16]:
with open(raw_output_filename, 'w', encoding='utf-8') as raw_output_file, \
     open(normal_output_filename, 'w', encoding='utf-8') as normal_output_file:
    
    for html_content in list_htmls:
        # Escape newlines and special characters for the raw version
        raw_string = repr(html_content)
        raw_output_file.write(raw_string + "\n")
        
        # Write the normal version without escaping
        normal_output_file.write(html_content + "\n")

For following procedures see notebook "Borgerforslag 2 - Data.ipynb" python notebook.